# TerraMind4특징과 RF

고정 TerraMind DEM384를 384→32→4 보조 신경망으로 압축합니다. 보조 분류 헤드는 표준화 물리26+DEM4→32→1이며 TRAIN으로 학습하고 최저 VALID BCE 가중치를 복원합니다. 이후 인코더를 고정하고 원 단위 물리26+DEM4=30변수로 RF를 학습합니다.

가공 파일에서 시작하여 학습·VALID·공통 TEST 평가·TEST SHAP를 한 파일에서 실행합니다. 다른 노트북의 커널 상태가 필요하지 않습니다. 모델 선택이나 임계값을 TEST 결과로 조정하지 않습니다. 표본 정의와 해석은 상위 README.md 및 DATA.md를 확인하세요.

## 1. 실행 설정

현재 환경에 requirements.txt의 모듈을 설치하고 같은 Python 커널을 선택합니다. `CASE`는 기본 `ABC`이며 `AB`·`AC`도 선택할 수 있습니다. MLP는 독립 최종 분류기로 사용하지 않습니다.

In [ ]:
from pathlib import Path
import sys

# 노트북 폴더·패키지 폴더·저장소 루트에서 열었을 때 공통 코드를 찾습니다.
cwd = Path.cwd()
PACKAGE = next(p for p in [cwd, cwd.parent, cwd/'experiments/16_model_comparison']
               if (p/'training.py').is_file())
PROJECT = next(p for p in [PACKAGE,*PACKAGE.parents] if (p/'ver3').is_dir())
sys.path.insert(0,str(PACKAGE))
from training import run_experiment

MODEL_NAME = 'terramind4_rf'
CASE = 'ABC'
DATA_DIR = PACKAGE/'data/processed'
OUTPUT_ROOT = PROJECT/'ver3/output/16/model_release/10_model_comparison'
LOG_ROOT = PROJECT/'ver3/log/16/model_release/10_model_comparison'
print(f'모델={MODEL_NAME}, 조합={CASE}')

## 2. 학습·성능평가·SHAP

파일 SHA-256·변수 순서·분할·표본키를 검증하고 TRAIN만으로 학습합니다. 압축형만 VALID BCE로 epoch를 선택합니다. 고정 모델의 VALID/TEST 예측 및 AP·ROC-AUC·Accuracy·Precision·Recall·F1을 저장합니다. SHAP 배경은 TRAIN100개이며 기여도 합산이 예측 확률과 일치하는지 확인합니다. 오래 걸리는 단계는 진행률이 표시됩니다.

In [ ]:
result = run_experiment(
    MODEL_NAME, CASE,
    data_dir=DATA_DIR, output_root=OUTPUT_ROOT, log_root=LOG_ROOT,
    run_shap=True,
)

## 3. 저장 결과 확인

VALID와 TEST 성능을 나누어 읽습니다. 압축 코드의 SHAP는 고도·경사 같은 물리량의 인과효과가 아닙니다. 누적 강우량 사이의 상관성을 고려하여 해석합니다.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

model_output = OUTPUT_ROOT/MODEL_NAME/CASE
metrics = json.loads((model_output/'metrics.json').read_text(encoding='utf-8'))
display(pd.DataFrame(metrics).T.round(4))
importance_path = model_output/'importance.csv'
if importance_path.exists():
    display(pd.read_csv(importance_path).head(10))
for name in ['roc_pr.png','shapbar.png']:
    if (model_output/name).exists():
        display(Image(filename=str(model_output/name)))